In [1]:
import torch
from torch import Tensor
import torch.nn.functional as F
import transformer_lens
from transformer_lens import utils
from transformer_lens import HookedTransformer, HookedTransformerConfig
import sae_lens
from sae_lens import SAE, ActivationsStore, HookedSAETransformer, LanguageModelSAERunnerConfig
from sae_lens.loading.pretrained_saes_directory import get_pretrained_saes_directory
from einops import einsum

from jaxtyping import Int, Float
from typing import List, Tuple, Optional, Literal
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import circuitsvis as cv
import plotly.express as px

from interp_utils import SAEStore
from interp_utils import get_layer_attributions, decomposed_head_outputs, decomposed_head_attribs
from plot_utils import visualize_transformer_attributions, get_and_visualize_model_attributions

In [2]:
model: HookedTransformer = HookedTransformer.from_pretrained("gpt2-small")

Loaded pretrained model gpt2-small into HookedTransformer


In [3]:
prompt = "Mitigating the risk of extinction from AI should be a global"
target = " priority"

In [4]:
model.eval()
with torch.no_grad():
    logits, cache = model.run_with_cache(prompt, remove_batch_dim=True)

In [5]:
# EMBEDDING STORY

In [6]:
tokens = model.to_str_tokens(prompt)
token_embeds = cache["hook_embed"]

In [7]:
# ATTENTION STORY

In [8]:
cache["blocks.0.hook_attn_out"].size(), cache["blocks.0.hook_resid_mid"].size()

(torch.Size([13, 768]), torch.Size([13, 768]))

In [9]:
head_outputs = decomposed_head_outputs(model, cache, with_distributed_bias=True)   # (n_layers, seq_len, n_heads, d_model)

head_decisions = einsum(
                    head_outputs,
                    model.W_U,
                    "n_layers seq_len n_heads d_model, d_model d_vocab -> n_layers seq_len n_heads d_vocab"
                    )

head_decisions.size()

torch.Size([12, 13, 12, 50257])

In [10]:
# MLP STORY

In [11]:
sae_model: HookedSAETransformer = HookedSAETransformer.from_pretrained("gpt2-small")

Loaded pretrained model gpt2-small into HookedTransformer


In [12]:
release_name = "gpt2-small-res-jb"
sae_store = SAEStore(release_name, device="cuda")

Loading SAEs:   0%|          | 0/12 [00:00<?, ?it/s]

/home/happyinterp/miniconda3/envs/mech-interp/lib/python3.11/site-packages/sae_lens/saes/sae.py:249: UserWarning: 
This SAE has non-empty model_from_pretrained_kwargs. 
For optimal performance, load the model like so:
model = HookedSAETransformer.from_pretrained_no_processing(..., **cfg.model_from_pretrained_kwargs)
  warnings.warn(


In [13]:
# 4 hooks per layer are added when run with saes
# hook_sae_input | hook_sae_acts_pre | hook_sae_acts_post | hook_sae_recons   : blocks.i.hook_resid_pre

_, sae_cache = sae_model.run_with_cache_with_saes(
    prompt,
    saes=[sae[1] for sae in list(sae_store.sae_dict.items())],
    remove_batch_dim=True,
)

In [14]:
# active neuron ratio
(sae_cache["blocks.0.hook_resid_pre.hook_sae_acts_post"] > 0.0001).sum() / (13 * 24576)

tensor(0.0007, device='cuda:0')

In [19]:
for layer in range(sae_model.cfg.n_layers):
    print(torch.topk(sae_cache[f"blocks.{layer}.hook_resid_pre.hook_sae_acts_post"][-1], k=3))

torch.return_types.topk(
values=tensor([2.6892, 0.7076, 0.6162], device='cuda:0'),
indices=tensor([14121,  7550, 18458], device='cuda:0'))
torch.return_types.topk(
values=tensor([30.1898,  6.2164,  3.5554], device='cuda:0'),
indices=tensor([23058, 21958, 24149], device='cuda:0'))
torch.return_types.topk(
values=tensor([32.4078,  2.4332,  2.0737], device='cuda:0'),
indices=tensor([16439, 15420,  1697], device='cuda:0'))
torch.return_types.topk(
values=tensor([35.0008,  1.9812,  1.8580], device='cuda:0'),
indices=tensor([11345,  7831, 12384], device='cuda:0'))
torch.return_types.topk(
values=tensor([38.9052,  3.2275,  2.7939], device='cuda:0'),
indices=tensor([14493, 20479, 24288], device='cuda:0'))
torch.return_types.topk(
values=tensor([44.9015,  4.9146,  4.6967], device='cuda:0'),
indices=tensor([   49, 18079, 20318], device='cuda:0'))
torch.return_types.topk(
values=tensor([44.4691,  4.7316,  3.2307], device='cuda:0'),
indices=tensor([2092, 3137, 2052], device='cuda:0'))
torch.return